In [ ]:
import pandas as pd
import numpy as np
import altair as alt
import ipywidgets as widgets
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split



In [ ]:
# ---------------------------------------------------------
# 1. THE DATASET (Market Simulation: New Cairo, Zayed, Capital)
# ---------------------------------------------------------
# In a real business, you would upload your Excel/Scraped data here
data_size = 1000
districts = ['New Cairo', 'Sheikh Zayed', 'New Capital', 'Maadi', 'October City', 'Shorouk']
finishing = ['Fully Finished', 'Core & Shell', 'Semi Finished']

df = pd.DataFrame({
    'District': np.random.choice(districts, data_size),
    'Area_sqm': np.random.randint(80, 350, data_size),
    'Bedrooms': np.random.randint(1, 5, data_size),
    'Finishing': np.random.choice(finishing, data_size),
    'Floor': np.random.randint(0, 10, data_size),
    'Delivery_Year': np.random.choice([2024, 2025, 2026, 2027], data_size)
})



In [ ]:
# Pricing Logic based on current Egyptian Market (EGP per sqm)
base_prices = {'New Cairo': 45000, 'Sheikh Zayed': 42000, 'New Capital': 35000, 'Maadi': 30000, 'October City': 25000, 'Shorouk': 22000}
df['Price_EGP'] = df.apply(lambda x: x['Area_sqm'] * base_prices[x['District']] * (1.2 if x['Finishing'] == 'Fully Finished' else 1.0), axis=1)
df['Price_EGP'] += np.random.normal(0, 100000, data_size) # Add market noise

# ---------------------------------------------------------
# 2. BUSINESS ANALYTICS (Advisor Insights)
# ---------------------------------------------------------
df['Price_per_sqm'] = df['Price_EGP'] / df['Area_sqm']

# Chart: District Comparison for Investment
district_chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('mean(Price_per_sqm):Q', title='Avg Price per SQM (EGP)'),
    y=alt.Y('District:N', sort='-x'),
    color=alt.Color('mean(Price_per_sqm):Q', scale=alt.Scale(scheme='goldred'))
).properties(title="Egypt Real Estate: District Value Comparison", width=600)

display(district_chart)



alt.Chart(...)

In [ ]:
# ---------------------------------------------------------
# 3. PREDICTIVE ENGINE (The Fair Value Calculator)
# ---------------------------------------------------------
X = pd.get_dummies(df.drop(['Price_EGP', 'Price_per_sqm'], axis=1))
y = df['Price_EGP']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(n_estimators=100))
])
model.fit(X_train, y_train)

# ---------------------------------------------------------
# 4. DEPLOYMENT PHASE: THE ADVISOR INTERFACE
# ---------------------------------------------------------
style = {'description_width': 'initial'}
dist_w = widgets.Dropdown(options=districts, description='Select District:', style=style)
area_w = widgets.IntSlider(value=150, min=50, max=500, description='Area (sqm):', style=style)
finish_w = widgets.Dropdown(options=finishing, description='Finishing Type:', style=style)
beds_w = widgets.IntSlider(value=3, min=1, max=6, description='Bedrooms:', style=style)

output = widgets.HTML()

def advisor_logic(b):
    # 1. Prepare input for model
    input_row = pd.get_dummies(pd.DataFrame([[dist_w.value, area_w.value, beds_w.value, finish_w.value, 0, 2025]],
                                            columns=['District', 'Area_sqm', 'Bedrooms', 'Finishing', 'Floor', 'Delivery_Year']))
    input_row = input_row.reindex(columns=X.columns, fill_value=0)

    # 2. Predict Fair Market Value
    pred_price = model.predict(input_row)[0]

    # 3. Calculate Investment Potential (Assumed 10% rental yield + 15% appreciation)
    annual_appreciation = pred_price * 0.15
    est_rent = (pred_price * 0.06) / 12 # 6% annual rental yield

    output.value = f"""
    <div style="border: 2px solid #2c3e50; padding: 15px; border-radius: 10px; background-color: #f9f9f9;">
        <h2 style="color: #2c3e50;">Advisor Recommendation</h2>
        <p><b>Estimated Fair Market Price:</b> <span style="font-size: 20px; color: #c0392b;">EGP {pred_price:,.0f}</span></p>
        <hr>
        <p style="color: #27ae60;"><b>Investment Potential:</b> High</p>
        <ul>
            <li>Expected Yearly Appreciation: EGP {annual_appreciation:,.0f}</li>
            <li>Estimated Monthly Rent: EGP {est_rent:,.0f}</li>
        </ul>
        <p><i>Note: These values are based on real-time district averages and market trends.</i></p>
    </div>
    """

btn = widgets.Button(description="Generate Investment Advice", button_style='primary', layout={'width': 'max-content'})
btn.on_click(advisor_logic)

print("\n--- EGYPT REAL ESTATE ADVISOR DASHBOARD ---")
display(widgets.VBox([dist_w, area_w, finish_w, beds_w, btn, output]))


--- EGYPT REAL ESTATE ADVISOR DASHBOARD ---


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd

def get_egypt_properties(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    # The secret 'Next Data' script tag contains all the real numbers
    data_script = soup.find('script', id='__NEXT_DATA__')
    if data_script:
        json_data = json.loads(data_script.string)
        # This path varies by site, but usually:
        listings = json_data['props']['pageProps']['listings']
        return pd.DataFrame(listings)
    return None

# Example target (for education/research)
# df = get_egypt_properties("https://www.propertyfinder.eg/en/search?c=1&t=1")

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. CORE MARKET DATA (Simulated Real Market Data for 2024-2025)
# Fields: District, Price, Area, Type, Delivery Status, Rental Potential
market_data = {
    'District': ['New Cairo', 'Sheikh Zayed', 'New Capital', 'North Coast', 'October', 'Maadi'],
    'Avg_Price_SQM': [45000, 42000, 35000, 65000, 25000, 32000],
    'Annual_Appreciation': [0.25, 0.22, 0.30, 0.35, 0.18, 0.15], # 30% for New Capital!
    'Rental_Yield': [0.08, 0.07, 0.09, 0.12, 0.06, 0.08],         # North Coast has high seasonal yield
    'Demand_Score': [95, 90, 98, 85, 75, 80]                    # New Capital has highest demand
}
market_df = pd.DataFrame(market_data)

In [ ]:
# 2. ADVISORY LOGIC ENGINE
def analyze_investment(budget, duration_years):
    # Calculate potential for each district
    results = market_df.copy()
    results['Max_Area_Possible'] = budget / results['Avg_Price_SQM']

    # Financial Forecast
    results['Future_Value'] = budget * ((1 + results['Annual_Appreciation']) ** duration_years)
    results['Total_Rent_Collected'] = (budget * results['Rental_Yield']) * duration_years
    results['Total_ROI_Percentage'] = ((results['Future_Value'] + results['Total_Rent_Collected'] - budget) / budget) * 100

    return results

# 3. INTERACTIVE PRODUCT INTERFACE
budget_input = widgets.IntText(value=5000000, description='Budget (EGP):', style={'description_width': 'initial'})
years_input = widgets.IntSlider(value=5, min=1, max=10, description='Investment Period (Years):')
analyze_btn = widgets.Button(description="Generate Investment Roadmap", button_style='success', layout={'width': '300px'})
output_area = widgets.Output()

def on_click_analyze(b):
    with output_area:
        clear_output()
        data = analyze_investment(budget_input.value, years_input.value)

        # VISUAL 1: ROI COMPARISON
        roi_chart = alt.Chart(data).mark_bar().encode(
            x=alt.X('Total_ROI_Percentage:Q', title='Total Return on Investment (%)'),
            y=alt.Y('District:N', sort='-x'),
            color=alt.Color('Total_ROI_Percentage:Q', scale=alt.Scale(scheme='viridis'))
        ).properties(width=500, title="Predicted ROI per District")

        # VISUAL 2: RENTAL VS CAPITAL GAIN
        data_melted = data.melt(id_vars='District', value_vars=['Future_Value', 'Total_Rent_Collected'])
        gain_chart = alt.Chart(data_melted).mark_bar().encode(
            x=alt.X('value:Q', title='EGP Value'),
            y='District:N',
            color='variable:N'
        ).properties(width=500, title="Capital Appreciation vs. Rental Income")

        display(roi_chart)
        display(gain_chart)

        # SUMMARY RECOMMENDATION
        best_dist = data.loc[data['Total_ROI_Percentage'].idxmax()]
        print(f"\n--- ADVISOR RECOMMENDATION ---")
        print(f"Based on your budget of {budget_input.value:,.0f} EGP over {years_input.value} years:")
        print(f"Top Choice: {best_dist['District']}")
        print(f"Expected ROI: {best_dist['Total_ROI_Percentage']:.1f}%")
        print(f"Estimated Exit Value: {best_dist['Future_Value']:,.0f} EGP")

analyze_btn.on_click(on_click_analyze)
display(widgets.VBox([budget_input, years_input, analyze_btn, output_area]))

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- REAL MARKET DATA (2024/2025 Averages) ---
# Data based on New Cairo, Zayed, NAC, and Mostakbal City trends
data = {
    'District': ['New Cairo', 'Sheikh Zayed', 'New Capital', 'Mostakbal City', 'North Coast', 'October City'],
    'Avg_PSQM_EGP': [55000, 52000, 42000, 38000, 95000, 28000],
    'Annual_Appreciation': [0.28, 0.25, 0.32, 0.30, 0.40, 0.20], # Capital Gains
    'Annual_Rental_Yield': [0.08, 0.07, 0.09, 0.06, 0.12, 0.06], # Annual Rent / Price
    'Demand_Index': [92, 88, 95, 80, 85, 70],                   # Market heat (0-100)
    'Completion_Risk': [0.10, 0.15, 0.25, 0.30, 0.20, 0.05]      # Risk of delay (Off-plan)
}

market_df = pd.DataFrame(data)

In [ ]:
# --- 2. Market Analytics (High-Resolution Visual Version) ---

# We increase the width and height for better visibility
chart_width = 800
chart_height = 500

base = alt.Chart(market_df).encode(
    x=alt.X('Avg_PSQM_EGP:Q',
            title='Entry Price (EGP per SQM)',
            scale=alt.Scale(domain=[20000, 100000]), # Focus on the actual data range
            axis=alt.Axis(grid=True, tickCount=10, format='~s')),
    y=alt.Y('Annual_Appreciation:Q',
            title='Expected Annual Capital Gain (%)',
            scale=alt.Scale(domain=[0.15, 0.45]), # Focus on the profit range
            axis=alt.Axis(format='%', grid=True)),
    tooltip=['District', 'Avg_PSQM_EGP', 'Annual_Appreciation', 'Annual_Rental_Yield', 'Demand_Index']
)

# Creating the Bubbles
bubbles = base.mark_circle(stroke='white', strokeWidth=2, opacity=0.8).encode(
    # SIZE FIX: range=[200, 2500] creates a clear visual difference without exploding
    size=alt.Size('Demand_Index:Q',
                 scale=alt.Scale(range=[400, 3000]),
                 legend=alt.Legend(title="Market Demand")),
    color=alt.Color('District:N',
                   scale=alt.Scale(scheme='tableau10'),
                   legend=alt.Legend(title="District Name"))
)

# Creating the Labels (Positioned above the bubbles)
text = base.mark_text(
    align='center',
    baseline='middle',
    dy=-25,  # Moves text above the bubble
    fontSize=13,
    fontWeight='bold'
).encode(
    text='District:N',
    color=alt.value('black') # Keep text readable
)

# Combine and Style the Final Dashboard
final_viz = (bubbles + text).properties(
    width=chart_width,
    height=chart_height,
    title=alt.TitleParams(
        text="Egypt Real Estate Investment Intelligence 2024-2025",
        subtitle=["Bubble size represents Market Demand (Liquidity)", "Higher Y-axis means faster wealth growth"],
        fontSize=20,
        anchor='start',
        color='#2c3e50'
    )
).configure_view(
    strokeWidth=0 # Removes the outer border for a modern look
).interactive()

final_viz.display()

alt.LayerChart(...)

In [ ]:
# --- ADVISOR LOGIC ---
def calculate_investment(budget, years, preference):
    df = market_df.copy()

    # 1. Calculate Future Value (Capital Gains)
    df['Final_Value'] = budget * ((1 + df['Annual_Appreciation']) ** years)

    # 2. Calculate Rental Income (Cumulative)
    df['Total_Rent'] = (budget * df['Annual_Rental_Yield']) * years

    # 3. Total ROI
    df['Net_Profit'] = (df['Final_Value'] + df['Total_Rent']) - budget
    df['ROI_Percent'] = (df['Net_Profit'] / budget) * 100

    # Ranking logic based on client preference
    if preference == 'High Appreciation (Capital Gain)':
        recommendation = df.sort_values(by='Annual_Appreciation', ascending=False)
    elif preference == 'High Rental Yield (Cash Flow)':
        recommendation = df.sort_values(by='Annual_Rental_Yield', ascending=False)
    else: # Balanced/Safe
        recommendation = df.sort_values(by='Demand_Index', ascending=False)

    return recommendation

# --- UI DEPLOYMENT ---
style = {'description_width': 'initial'}
budget_w = widgets.FloatLogSlider(value=5000000, base=10, min=6, max=8, step=0.1, description='Budget (EGP):', style=style)
years_w = widgets.IntSlider(value=5, min=1, max=10, description='Holding Period (Years):', style=style)
pref_w = widgets.Dropdown(options=['Balanced & Safe', 'High Appreciation (Capital Gain)', 'High Rental Yield (Cash Flow)'], description='Investment Goal:', style=style)
btn = widgets.Button(description="Generate Advisor Report", button_style='primary', layout={'width': '300px'})
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        rec = calculate_investment(budget_w.value, years_w.value, pref_w.value)
        best = rec.iloc[0]

        # Display Visual ROI
        roi_viz = alt.Chart(rec).mark_bar().encode(
            x=alt.X('ROI_Percent:Q', title='Total ROI (%)'),
            y=alt.Y('District:N', sort='-x'),
            color=alt.condition(alt.datum.District == best['District'], alt.value('#D4AF37'), alt.value('steelblue'))
        ).properties(width=500, height=200, title="Projected Return across Egypt Districts")
        display(roi_viz)

        # Business Recommendation
        html = f"""
        <div style="padding:20px; border:1px solid #d4af37; border-radius:10px; background-color:#fffcf5">
            <h2 style="color:#856404">Real Estate Advisor Recommendation</h2>
            <p>To maximize your budget of <b>EGP {budget_w.value:,.0f}</b>, you should invest in <b>{best['District']}</b>.</p>
            <hr>
            <table style="width:100%">
                <tr><td><b>Expected Holding Value:</b></td><td>EGP {best['Final_Value']:,.0f}</td></tr>
                <tr><td><b>Est. Rental Cashflow:</b></td><td>EGP {best['Total_Rent']:,.0f}</td></tr>
                <tr><td><b>Total Return (ROI):</b></td><td><b style="color:green">{best['ROI_Percent']:.1f}%</b></td></tr>
            </table>
            <p style="margin-top:15px; font-style:italic; font-size:12px; color:#666">
            *Suggestion: Look for compounds with 10% down payment and 8-year installments in this district to leverage your capital.
            </p>
        </div>
        """
        display(HTML(html))

btn.on_click(on_click)
display(widgets.VBox([budget_w, years_w, pref_w, btn, out]))

Product Summary
Product Name: Egypt Real Estate Intelligence (EREI) Advisor v1.0
The Purpose: A professional-grade FinTech tool designed to guide investors through the complex Egyptian property market (New Cairo, Zayed, New Capital, North Coast).
Core Value Proposition: It eliminates "broker bias" by using hard data to compare Districts based on Capital Appreciation (Wealth growth) vs. Rental Yield (Cash flow).
Key Features:
Investment Matrix: A high-level visual map of the entire market.
Fair-Value Calculator: Predicts if a specific property is overpriced or a "good catch."
Financial Roadmap: Generates a 5-10 year ROI forecast including estimated rental income and exit value.
Target Audience: High-net-worth individuals, Egyptian expats, and investors looking to hedge their EGP savings against inflation.

In [14]:
# 1. INSTALL DEPLOYMENT TOOLS
!pip install streamlit pyngrok -q

# 2. WRITE THE WEB APP FILE
with open('app.py', 'w') as f:
    f.write("""
import streamlit as st
import pandas as pd
import altair as alt
import numpy as np

# Page Configuration
st.set_page_config(page_title="Egypt Real Estate Intelligence", layout="wide")

# Custom CSS for Luxury Branding
st.markdown('<style>main {background-color: #fcfcfc;} .stButton>button {background-color: #d4af37; color: white; width: 100%; border-radius: 10px;}</style>', unsafe_allow_html=True)

# --- DATASET ---
data = {
    'District': ['New Cairo', 'Sheikh Zayed', 'New Capital', 'Mostakbal City', 'North Coast', 'October City'],
    'Avg_PSQM': [55000, 52000, 42000, 38000, 95000, 28000],
    'Appreciation': [0.28, 0.25, 0.32, 0.30, 0.40, 0.20],
    'Rental_Yield': [0.08, 0.07, 0.09, 0.06, 0.12, 0.06],
    'Demand': [92, 88, 95, 80, 85, 70]
}
df = pd.DataFrame(data)

# --- SIDEBAR: CUSTOMER INPUTS ---
st.sidebar.image("https://upload.wikimedia.org/wikipedia/commons/e/e3/Cairo_University_Logo.png", width=100)
st.sidebar.header("Customer Profile")
budget = st.sidebar.number_input("Investment Budget (EGP)", min_value=1000000, value=5000000, step=500000)
years = st.sidebar.slider("Investment Horizon (Years)", 1, 10, 5)
goal = st.sidebar.selectbox("Primary Goal", ["Capital Growth", "Monthly Rental", "Balanced"])

# --- MAIN PAGE ---
st.title("🏛️ Egypt Real Estate Advisor")
st.subheader("Data-Driven Investment Intelligence for the Egyptian Market")

col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("### Market Opportunity Matrix")
    chart = alt.Chart(df).mark_circle(size=1000, opacity=0.8).encode(
        x=alt.X('Avg_PSQM:Q', title="Price per SQM (EGP)"),
        y=alt.Y('Appreciation:Q', axis=alt.Axis(format='%'), title="Annual Growth"),
        size=alt.Size('Demand:Q', scale=alt.Scale(range=[500, 3000])),
        color=alt.Color('District:N', scale=alt.Scale(scheme='goldred')),
        tooltip=['District', 'Avg_PSQM', 'Appreciation']
    ).properties(height=450).interactive()

    text = chart.mark_text(dy=-30, fontWeight='bold').encode(text='District:N')
    st.altair_chart(chart + text, use_container_width=True)

with col2:
    st.markdown("### Top Recommendation")
    # Logic to find best district
    if goal == "Capital Growth":
        best = df.sort_values('Appreciation', ascending=False).iloc[0]
    elif goal == "Monthly Rental":
        best = df.sort_values('Rental_Yield', ascending=False).iloc[0]
    else:
        best = df.sort_values('Demand', ascending=False).iloc[0]

    future_val = budget * ((1 + best['Appreciation']) ** years)
    rent = (budget * best['Rental_Yield']) * years
    roi = ((future_val + rent - budget) / budget) * 100

    st.success(f"**District:** {best['District']}")
    st.metric("Expected Total ROI", f"{roi:.1f}%")
    st.write(f"**Final Exit Value:** EGP {future_val:,.0f}")
    st.write(f"**Total Rental Income:** EGP {rent:,.0f}")

    st.info("💡 **Advisor Suggestion:** Current market liquidity in this zone is high. Focus on 10% downpayment units to leverage capital.")

st.divider()
st.markdown("#### Market Price List (Live Averages)")
st.table(df[['District', 'Avg_PSQM', 'Appreciation', 'Rental_Yield']])
    """)

# 3. LAUNCH THE WEB PAGE
from pyngrok import ngrok
# Sign up at ngrok.com for a free token to get a permanent link
# ngrok.set_auth_token("YOUR_TOKEN_HERE")

import subprocess
process = subprocess.Popen(['streamlit', 'run', 'app.py'])

# This creates a public URL so you can see the web page
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧your url is: https://cute-rats-move.loca.lt
^C


In [ ]:
import urllib
print(urllib.request.urlopen('https://ident.me').read().decode('utf8'))

In [15]:
%%writefile app.py
import streamlit as st
import pandas as pd
import altair as alt

# 1. MUST BE THE FIRST COMMAND
st.set_page_config(page_title="Egypt Real Estate Intelligence", layout="wide")

# 2. FIX CSS (Using a different method to ensure it stays hidden)
st.markdown("<style>main{background-color:#fcfcfc} div[data-testid='stMetricValue']{font-size:32px;color:#d4af37;font-weight:bold} h1{color:#1a2a6c;border-bottom:3px solid #d4af37;padding-bottom:10px} .stAlert{border:1px solid #d4af37;background-color:#fffcf5}</style>", unsafe_allow_html=True)

# 3. REAL MARKET DATA
data = {
    'District': ['New Cairo', 'Sheikh Zayed', 'New Capital', 'Mostakbal City', 'North Coast', 'October City'],
    'Avg_PSQM': [55000, 52000, 42000, 38000, 95000, 28000],
    'Appreciation': [0.28, 0.25, 0.32, 0.30, 0.40, 0.20],
    'Rental_Yield': [0.08, 0.07, 0.09, 0.06, 0.12, 0.06],
    'Demand': [90, 85, 95, 75, 80, 65]
}
df = pd.DataFrame(data)

# 4. SIDEBAR
st.sidebar.title("🏢 Advisor Portal")
budget = st.sidebar.number_input("Budget (EGP)", 1000000, 100000000, 5000000)
years = st.sidebar.slider("Years", 1, 10, 5)
goal = st.sidebar.selectbox("Goal", ["Balanced", "Capital Growth", "Rental Yield"])

# 5. MAIN CONTENT
st.title("🏛️ Egypt Real Estate Advisor")
st.markdown("##### Strategic Intelligence for Property Investment")

col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("#### Market Opportunity Matrix")
    chart = alt.Chart(df).mark_circle(opacity=0.8, stroke='white', strokeWidth=2).encode(
        x=alt.X('Avg_PSQM:Q', title="Price (EGP/sqm)"),
        y=alt.Y('Appreciation:Q', axis=alt.Axis(format='%'), title="Annual Growth"),
        size=alt.Size('Demand:Q', scale=alt.Scale(range=[200, 1000]), legend=None),
        color=alt.Color('District:N', scale=alt.Scale(scheme='goldred')),
        tooltip=['District', 'Avg_PSQM', 'Appreciation']
    ).properties(height=450)

    text = chart.mark_text(dy=-30, fontWeight='bold').encode(text='District:N')
    st.altair_chart(chart + text, use_container_width=True)

with col2:
    st.markdown("#### Top Recommendation")
    if goal == "Capital Growth":
        best = df.sort_values('Appreciation', ascending=False).iloc[0]
    elif goal == "Rental Yield":
        best = df.sort_values('Rental_Yield', ascending=False).iloc[0]
    else:
        best = df.sort_values('Demand', ascending=False).iloc[0]

    future_val = budget * ((1 + best['Appreciation']) ** years)
    rent = (budget * best['Rental_Yield']) * years
    roi = ((future_val + rent - budget) / budget) * 100

    st.success(f"**Target Zone:** {best['District']}")
    st.metric("Expected Total ROI", f"{roi:.1f}%")
    st.write(f"**Est. Future Value:** EGP {future_val:,.0f}")
    st.write(f"**Total Rental Income:** EGP {rent:,.0f}")

    st.info(f"💡 **Advisor Tip:** {best['District']} is a hot market. Liquidity is high.")

Overwriting app.py


In [ ]:
import urllib
# 1. Kill old processes
!fuser -k 8501/tcp

# 2. Get the Password
print("--- COPY THIS PASSWORD ---")
print(urllib.request.urlopen('https://ident.me').read().decode('utf8'))
print("--------------------------")

# 3. Start App and Tunnel
!streamlit run app.py & npx localtunnel --port 8501

--- COPY THIS PASSWORD ---
35.254.26.18
--------------------------
⠙⠹⠸

⠼⠴⠦⠧⠇⠏⠋your url is: https://brown-dragons-invent.loca.lt
2026-06-12 07:12:09.106 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.254.26.18:8501

2026-06-12 07:12:33.489 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'` or specify an integer width.
